In [1]:
import matplotlib.pyplot as plt
import xarray as xr
from sklearn.metrics import r2_score
from pathlib import Path
import pandas as pd
import os

In [17]:
# ------------ Paths ------------
# basins_path = Path("./basins_excluded.txt")
basins_ws_path = Path("./basins_subset_test.txt")
original_caravan_path = "/inputs/data_updated_2/time_series"
camels_timeseries = Path("/inputs/basin_dataset_public_v1p2/basin_mean_forcing/daymet") #US
chirps_dir = Path("../CHIRPS_3.0/precip_timeseries_chirpsv3_extract") 
mswep_dir = Path("../MSWEP/precip_timeseries_mswep_weighted_2")

In [3]:
def open_zarr(path):
  return xr.open_zarr(
      store=path, chunks=None, storage_options=dict(token='anon')
  )

In [4]:
products = [
    'CPC',
    'IMERG',
    'CHIRPS',
    'ERA5_LAND',
    'CHIRPS_GEFS',
    'HRES',
    'GRAPHCAST',
]
zarr_path_template = 'gs://caravan-multimet/v1.1/{}/timeseries.zarr/'

product_to_dataset = {
    product: open_zarr(zarr_path_template.format(product))
    for product in products
}

In [5]:
# Print summary of the different data products.
for product_name, ds in product_to_dataset.items():
  print(f'{product_name}:\n - Number of basins {len(ds["basin"])}')
  print(f' - Start time {ds["date"].values[0]}')
  print(f' - End time {ds["date"].values[-1]}')
  if "lead_time" in ds.coords:
    print(f' - Number of forecast time steps {len(ds["lead_time"])}')
  bands = "".join([f"   - {str(x)}\n" for x in ds.data_vars])
  print(f" - Bands: \n{bands}")

CPC:
 - Number of basins 22492
 - Start time 1979-01-01T00:00:00.000000000
 - End time 2024-07-31T00:00:00.000000000
 - Bands: 
   - cpc_precipitation

IMERG:
 - Number of basins 22492
 - Start time 2000-06-01T00:00:00.000000000
 - End time 2024-10-31T00:00:00.000000000
 - Bands: 
   - imerg_precipitation

CHIRPS:
 - Number of basins 18655
 - Start time 1981-01-01T00:00:00.000000000
 - End time 2024-07-30T00:00:00.000000000
 - Bands: 
   - chirps_precipitation

ERA5_LAND:
 - Number of basins 22485
 - Start time 1950-01-01T00:00:00.000000000
 - End time 2024-10-31T00:00:00.000000000
 - Bands: 
   - era5land_dewpoint_temperature_2m
   - era5land_potential_evaporation_DEPRECATED
   - era5land_potential_evaporation_FAO_PENMAN_MONTEITH
   - era5land_snow_depth_water_equivalent
   - era5land_surface_net_solar_radiation
   - era5land_surface_net_thermal_radiation
   - era5land_surface_pressure
   - era5land_temperature_2m
   - era5land_total_precipitation
   - era5land_u_component_of_wind_10m

In [6]:
with open(basins_ws_path, "r") as f:
    basins = f.read().splitlines()
len(basins)

127

In [7]:
# Which data?
product = "GRAPHCAST"
band = "graphcast_temperature_2m"

# Which basin and time period?
basin = "camelsaus_102101A"
time_period = slice("2023-01", "2023-02")

ds = product_to_dataset[product]
# Only with the .compute() in this line, data is actually loaded into memory.
da = ds[band].sel(basin=basin, date=time_period) #.compute()
da

<xarray.DataArray 'graphcast_temperature_2m' (date: 59, lead_time: 10)> Size: 2kB
[590 values with dtype=float32]
Coordinates:
    basin      <U22 88B 'camelsaus_102101A'
  * date       (date) datetime64[ns] 472B 2023-01-01 2023-01-02 ... 2023-02-28
  * lead_time  (lead_time) timedelta64[ns] 80B 1 days 2 days ... 9 days 10 days

In [9]:
product_and_bands= {
    # "CHIRPS": ["chirps_precipitation"],
    "ERA5_LAND": ["era5land_total_precipitation"]}

time_period = slice("1989-01-01", "2019-12-31")

In [10]:

original_vars = [
    "total_precipitation_sum",
    "temperature_2m_max",
    "temperature_2m_min",
    "surface_net_solar_radiation_mean",
    "streamflow"
]

for basin in basins:
    # Loop over products, select bands/basins/time, merge into one dataset
    datasets = []
    for product, bands in product_and_bands.items():
        ds = product_to_dataset[product]
        ds_sel = ds[bands].sel(basin=basin, date=time_period).compute()
        datasets.append(ds_sel)

    # Open original caravan dataset and extract variables
    original = xr.open_dataset(f"{original_caravan_path}/{basin}.nc")
    original_sel = original[original_vars].sel(date=time_period)
    datasets.append(original_sel)

    merged = xr.merge(datasets)

    # Save to NetCDF
    output_path = Path(f"./data/time_series/{basin}.nc")
    merged.to_netcdf(output_path)
    print(f"Saved {basin} to {output_path}")


Saved camels_01411300 to data/time_series/camels_01411300.nc
Saved camels_01466500 to data/time_series/camels_01466500.nc
Saved camels_01487000 to data/time_series/camels_01487000.nc
Saved camels_01638480 to data/time_series/camels_01638480.nc
Saved camels_01644000 to data/time_series/camels_01644000.nc
Saved camels_01666500 to data/time_series/camels_01666500.nc
Saved camels_01667500 to data/time_series/camels_01667500.nc
Saved camels_01669000 to data/time_series/camels_01669000.nc
Saved camels_01669520 to data/time_series/camels_01669520.nc
Saved camels_02027000 to data/time_series/camels_02027000.nc
Saved camels_02038850 to data/time_series/camels_02038850.nc
Saved camels_02046000 to data/time_series/camels_02046000.nc
Saved camels_02053200 to data/time_series/camels_02053200.nc
Saved camels_02053800 to data/time_series/camels_02053800.nc
Saved camels_02055100 to data/time_series/camels_02055100.nc
Saved camels_02064000 to data/time_series/camels_02064000.nc
Saved camels_02065500 to

In [11]:
nc_file=xr.open_dataset("./data/time_series/camels_01411300.nc")
nc_file

<xarray.Dataset> Size: 362kB
Dimensions:                           (date: 11322)
Coordinates:
    basin                             <U15 60B ...
  * date                              (date) datetime64[ns] 91kB 1989-01-01 ....
Data variables:
    era5land_total_precipitation      (date) float32 45kB ...
    total_precipitation_sum           (date) float32 45kB ...
    temperature_2m_max                (date) float32 45kB ...
    temperature_2m_min                (date) float32 45kB ...
    surface_net_solar_radiation_mean  (date) float32 45kB ...
    streamflow                        (date) float32 45kB ...
Attributes:
    Citation:  Muñoz Sabater, J. (2019): ERA5-Land hourly data from 1950 to p...
    License:   https://cds.climate.copernicus.eu/api/v2/terms/static/licence-...
    Product:   ERA5-Land
    Released:  2024-11-18
    Sources:   All forcing and state variables are derived from ERA5-Land hou...
    Units:     dewpoint_temperature_2m: Dew point temperature [°C]\npotential...
    Version:   1.1

In [12]:
# basin="camels_02011400"
# precip=pd.read_csv(f"./mswep_precip_timeseries/{basin}_precip.csv")
# precip.head()

# Adding CAMELS

In [13]:
for basin_name in basins:
    basin_id = basin_name.replace("camels_", "")

    try:
        # --- Load nc file ---
        nc_path = f"./data/time_series/{basin_name}.nc"
        ds = xr.open_dataset(nc_path)

        # --- Load CAMELS txt ---
        file_path = next(camels_timeseries.rglob(f"*{basin_id}*.txt"))
        camels = pd.read_csv(file_path, sep=r"\s+", skiprows=3)

        # Build datetime
        camels["date"] = pd.to_datetime(
            camels[["Year", "Mnth", "Day"]]
            .rename(columns={"Year": "year", "Mnth": "month", "Day": "day"})
        )

        # --- Create DataArray ---
        camels_da = xr.DataArray(
            data=camels["prcp(mm/day)"].values,
            dims=["date"],
            coords={"date": camels["date"].values},
        )

        # --- Align to nc dates ONLY ---
        camels_da = camels_da.reindex(date=ds["date"])

        # --- Add to dataset ---
        ds["camels_precipitation"] = camels_da

        # --- Safe save ---
        ds.to_netcdf(nc_path + ".tmp")
        ds.close()
        os.replace(nc_path + ".tmp", nc_path)

        print(f"Added camels_precipitation to {basin_name}")

    except Exception as e:
        print(f"Failed for {basin_name}: {e}") 

Added camels_precipitation to camels_01411300
Added camels_precipitation to camels_01466500
Added camels_precipitation to camels_01487000
Added camels_precipitation to camels_01638480
Added camels_precipitation to camels_01644000
Added camels_precipitation to camels_01666500
Added camels_precipitation to camels_01667500
Added camels_precipitation to camels_01669000
Added camels_precipitation to camels_01669520
Added camels_precipitation to camels_02027000
Added camels_precipitation to camels_02038850
Added camels_precipitation to camels_02046000
Added camels_precipitation to camels_02053200
Added camels_precipitation to camels_02053800
Added camels_precipitation to camels_02055100
Added camels_precipitation to camels_02064000
Added camels_precipitation to camels_02065500
Added camels_precipitation to camels_02081500
Added camels_precipitation to camels_02082950
Added camels_precipitation to camels_02092500
Added camels_precipitation to camels_02108000
Added camels_precipitation to came

# Addig CHIRPS

In [15]:
for basin_name in basins:

    try:
        # --- Load nc file ---
        nc_path = f"./data/time_series/{basin_name}.nc"
        ds = xr.open_dataset(nc_path)

        # Load and prepare chirps_v3
        chirps_v3 = pd.read_csv(chirps_dir / f"{basin_name}_precip.csv", 
                                index_col=0, parse_dates=True)
        chirps_v3.index.name = "date"
        chirps_v3.columns = ["chirps_precipitation"]

        # Convert to xarray and align to ds dates
        chirps_v3_da = chirps_v3["chirps_precipitation"].to_xarray()
        chirps_v3_da = chirps_v3_da.sel(date=ds.date)

        # Add to dataset
        ds["chirps_precipitation"] = chirps_v3_da


        # --- Safe save ---
        ds.to_netcdf(nc_path + ".tmp")
        ds.close()
        os.replace(nc_path + ".tmp", nc_path)

        print(f"Added chirps_precipitation to {basin_name}")

    except Exception as e:
        print(f"Failed for {basin_name}: {e}") 

Added chirps_precipitation to camels_01411300
Added chirps_precipitation to camels_01466500
Added chirps_precipitation to camels_01487000
Added chirps_precipitation to camels_01638480
Added chirps_precipitation to camels_01644000
Added chirps_precipitation to camels_01666500
Added chirps_precipitation to camels_01667500
Added chirps_precipitation to camels_01669000
Added chirps_precipitation to camels_01669520
Added chirps_precipitation to camels_02027000
Added chirps_precipitation to camels_02038850
Added chirps_precipitation to camels_02046000
Added chirps_precipitation to camels_02053200
Added chirps_precipitation to camels_02053800
Added chirps_precipitation to camels_02055100
Added chirps_precipitation to camels_02064000
Added chirps_precipitation to camels_02065500
Added chirps_precipitation to camels_02081500
Added chirps_precipitation to camels_02082950
Added chirps_precipitation to camels_02092500
Added chirps_precipitation to camels_02108000
Added chirps_precipitation to came

In [16]:
nc_file=xr.open_dataset("./data/time_series/camels_01411300.nc")
nc_file

<xarray.Dataset> Size: 544kB
Dimensions:                           (date: 11322)
Coordinates:
    basin                             <U15 60B ...
  * date                              (date) datetime64[ns] 91kB 1989-01-01 ....
Data variables:
    era5land_total_precipitation      (date) float32 45kB ...
    total_precipitation_sum           (date) float32 45kB ...
    temperature_2m_max                (date) float32 45kB ...
    temperature_2m_min                (date) float32 45kB ...
    surface_net_solar_radiation_mean  (date) float32 45kB ...
    streamflow                        (date) float32 45kB ...
    camels_precipitation              (date) float64 91kB ...
    chirps_precipitation              (date) float64 91kB ...
Attributes:
    Citation:  Muñoz Sabater, J. (2019): ERA5-Land hourly data from 1950 to p...
    License:   https://cds.climate.copernicus.eu/api/v2/terms/static/licence-...
    Product:   ERA5-Land
    Released:  2024-11-18
    Sources:   All forcing and state variables are derived from ERA5-Land hou...
    Units:     dewpoint_temperature_2m: Dew point temperature [°C]\npotential...
    Version:   1.1

In [32]:
# precip_da = xr.DataArray(
#     data=precip["precip_mm"].values,
#     dims=["date"],
#     coords={"date": pd.to_datetime(precip["date"]).values},
# )
# nc_file["mswep_precipitation"] = precip_da
# nc_file

# Addig MSWEP

In [18]:
for basin_name in basins:

    try:
        # --- Load nc file ---
        nc_path = f"./data/time_series/{basin_name}.nc"
        ds = xr.open_dataset(nc_path)

        # Load and prepare chirps_v3
        mswep = pd.read_csv(mswep_dir / f"{basin_name}_precip.csv", 
                                index_col=0, parse_dates=True)
        mswep.index.name = "date"
        mswep.columns = ["mswep_precipitation"]

        # Convert to xarray and align to ds dates
        mswep_da = mswep["mswep_precipitation"].to_xarray()
        mswep_da = mswep_da.sel(date=ds.date)

        # Add to dataset
        ds["mswep_precipitation"] = mswep_da


        # --- Safe save ---
        ds.to_netcdf(nc_path + ".tmp")
        ds.close()
        os.replace(nc_path + ".tmp", nc_path)

        print(f"Added mswep_precipitation to {basin_name}")

    except Exception as e:
        print(f"Failed for {basin_name}: {e}") 

Added mswep_precipitation to camels_01411300
Added mswep_precipitation to camels_01466500
Added mswep_precipitation to camels_01487000
Added mswep_precipitation to camels_01638480
Added mswep_precipitation to camels_01644000
Added mswep_precipitation to camels_01666500
Added mswep_precipitation to camels_01667500
Added mswep_precipitation to camels_01669000
Added mswep_precipitation to camels_01669520
Added mswep_precipitation to camels_02027000
Added mswep_precipitation to camels_02038850
Added mswep_precipitation to camels_02046000
Added mswep_precipitation to camels_02053200
Added mswep_precipitation to camels_02053800
Added mswep_precipitation to camels_02055100
Added mswep_precipitation to camels_02064000
Added mswep_precipitation to camels_02065500
Added mswep_precipitation to camels_02081500
Added mswep_precipitation to camels_02082950
Added mswep_precipitation to camels_02092500
Added mswep_precipitation to camels_02108000
Added mswep_precipitation to camels_02118500
Added mswe

In [19]:
xr.open_dataset("./data/time_series/camels_01411300.nc")

<xarray.Dataset> Size: 634kB
Dimensions:                           (date: 11322)
Coordinates:
    basin                             <U15 60B ...
  * date                              (date) datetime64[ns] 91kB 1989-01-01 ....
Data variables:
    era5land_total_precipitation      (date) float32 45kB ...
    total_precipitation_sum           (date) float32 45kB ...
    temperature_2m_max                (date) float32 45kB ...
    temperature_2m_min                (date) float32 45kB ...
    surface_net_solar_radiation_mean  (date) float32 45kB ...
    streamflow                        (date) float32 45kB ...
    camels_precipitation              (date) float64 91kB ...
    chirps_precipitation              (date) float64 91kB ...
    mswep_precipitation               (date) float64 91kB ...
Attributes:
    Citation:  Muñoz Sabater, J. (2019): ERA5-Land hourly data from 1950 to p...
    License:   https://cds.climate.copernicus.eu/api/v2/terms/static/licence-...
    Product:   ERA5-Land
    Released:  2024-11-18
    Sources:   All forcing and state variables are derived from ERA5-Land hou...
    Units:     dewpoint_temperature_2m: Dew point temperature [°C]\npotential...
    Version:   1.1